In [2]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv
import urllib.parse
import os
import clickhouse_connect
import numpy as np
import pandas as pd
import datetime as dt
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

orange = '#FF670F'
pink = '#FC006D'
blue = '#006CED'
green = '#2BD67C'
red = '#FF5D5D'
black = '#2B3034'

#tables
grdf_all='datamart_grdf.dataset_grdf_all'
releves='datawarehouse_prod.fact_payload'
clients='datawarehouse_prod.dim_entity'
equipements='datawarehouse_prod.dim_device'
capteurs='datawarehouse_prod.dim_sensor'
patrimoine='datawarehouse_prod.device_patrimonial_data'

source="grdf"
maxi = 10000000000000

load_dotenv()
host = urllib.parse.quote_plus(os.environ.get("clk_prod"))
username = urllib.parse.quote_plus(os.environ.get("USER_NAME"))
password = urllib.parse.quote_plus(os.environ.get('clk_prod_pass'))
client = clickhouse_connect.get_client(host=host, username=username, 
                                       port = 443, password=password)

In [115]:
df = client.query_df(f"""
SELECT distinct provider_name, contractreference
FROM datawarehouse_prod.fact_nrj_month_data_water_bill
settings max_rows_to_read={maxi};""")
df = df[~df.contractreference.isna()]

In [116]:
df.loc[:, 'FT1'] = df.contractreference.apply(lambda x: len(x)) # taille du nom
df.loc[:, 'FT2'] = df.contractreference.apply(lambda x: len(re.findall(r"-", x))) # nombre de tirets
df.loc[:, 'FT3'] = df.contractreference.apply(lambda x: len(re.findall(r"[0-9]", x))) # nombre de chiffres
df.loc[:, 'FT4'] = df.contractreference.apply(lambda x: len(re.findall(r"[a-z]", x))) # nombre de lettres minuscules
df.loc[:, 'FT5'] = df.contractreference.apply(lambda x: len(re.split(r"[-.]", x)[0])) # taille du groupe 1 (taille de la chaine si pas de tiret)
df.loc[:, 'FT6'] = df.contractreference.apply(lambda x: len(re.split(r"[-.]", x)[1]) if len(re.split(r"[-.]", x)) > 1  else 0)
df.loc[:, 'FT7'] = df.contractreference.apply(lambda x: len(re.split(r"[-.]", x)[2]) if len(re.split(r"[-.]", x)) > 2 else 0)
df.loc[:, 'FT8'] = df.contractreference.apply(lambda x: len(re.split(r"[-.]", x))) # nombre de groupes séparés par - ou .
df.loc[:, 'FT9'] = df.contractreference.apply(lambda x: len(re.findall(r"[A-Z]", x))) # nombre de lettres majuscules
df.loc[:, 'FT10'] = df.contractreference.apply(lambda x: len(re.split(r"[-.]", x)[3]) if len(re.split(r"[-.]", x)) > 3 else 0)
df.loc[:, 'FT11'] = df.contractreference.apply(lambda x: len(re.findall(r"\.", x))) # nombre de points

In [117]:
Z = linkage(df.drop(['provider_name', 'contractreference'], axis=1).values, method='ward')  # 'ward', 'single', 'complete', 'average'...
clusters = fcluster(Z, t=15, criterion='maxclust')
df["cluster15"] = clusters

In [82]:
df.groupby(['provider_name', 'cluster15'], as_index=False).contractreference.agg(['count', 'max']).sort_values('count').head(20)

,provider_name,cluster15,count,max
204,Veolia,5,156,92012
33,Colmarienne des eaux,15,159,98-9927788009
88,Eau du Grand Lyon,4,160,2113430
74,Eau d'Azur,8,196,OLD_1240822
215,Veolia Eau,1,211,1475701900433101
120,"L""Eau d'Ile de France",7,216,956165358
73,Eau d'Azur,4,236,4939260
218,Veolia Eau,5,247,92012
92,Eau et Force,13,272,65-114482-00
94,Eau et Force,15,283,98-9936220000


In [83]:
df.groupby(['cluster15'], as_index=False).contractreference.agg(['count', 'max']).sort_values('count')

,cluster15,count,max
2,3,1,1082031Nsontassuréesengestionpubliquepar:
10,11,9,€tod'lk-<k-FrMr\Cl825482643
8,9,16,SociétéEUROPCARFRANCE
1,2,20,2800000000005000000001
11,12,64,930000.123.00100.05
5,6,70,VILLE
0,1,270,768100151067801
7,8,354,TISSEO2022
4,5,624,V0977
12,13,640,OLD_20-080376-00


In [119]:
# avec 20 clusters 
clusters = fcluster(Z, t=20, criterion='maxclust')
df["cluster20"] = clusters

In [87]:
df.groupby(['cluster20'], as_index=False).contractreference.agg(['count', 'max']).sort_values('count')

,cluster20,count,max
3,4,1,1082031Nsontassuréesengestionpubliquepar:
2,3,2,016103/3218016706001J19MU254848Z
14,15,6,28229.340.0000004173.000.002
13,14,9,€tod'lk-<k-FrMr\Cl825482643
11,12,16,SociétéEUROPCARFRANCE
1,2,18,2800000000005000000001
16,17,28,2025-EA-00-2025009891
15,16,30,930000.123.00100.05
7,8,70,VILLE
9,10,179,A0090190201


In [120]:
clusters = fcluster(Z, t=20, criterion='distance')
df['cluster_dist_20'] = clusters

In [103]:
df.groupby(['cluster_dist_20'], as_index=False).contractreference.agg(['count', 'max']).sort_values('count')

,cluster_dist_20,count,max
3,4,1,1082031Nsontassuréesengestionpubliquepar:
2,3,2,016103/3218016706001J19MU254848Z
15,16,3,EAU-LESPONTSDECE
14,15,4,RCU_INRIA_RENNES
18,19,6,28229.340.0000004173.000.002
17,18,9,€tod'lk-<k-FrMr\Cl825482643
13,14,9,SociétéEUROPCARFRANCE
8,9,12,VILLE
1,2,18,2800000000005000000001
22,23,25,OLD_20-080376-00


In [105]:
df[df.cluster_dist_20==12]

,provider_name,contractreference,FT1,FT2,FT3,FT4,FT5,FT6,FT7,FT8,FT9,FT10,FT11,cluster15,cluster20,cluster_dist_20
4229,EXCEL HISTO CAGNE SUR MER,OLD_3502333G,12,0,7,0,12,0,0,1,4,0,0,8,11,12
4230,EXCEL HISTO CAGNE SUR MER,OLD_3505920Y,12,0,7,0,12,0,0,1,4,0,0,8,11,12
4231,EXCEL HISTO CAGNE SUR MER,OLD_3506094E,12,0,7,0,12,0,0,1,4,0,0,8,11,12
4232,EXCEL HISTO CAGNE SUR MER,OLD_3510561R,12,0,7,0,12,0,0,1,4,0,0,8,11,12
4233,EXCEL HISTO CAGNE SUR MER,OLD_3510609T,12,0,7,0,12,0,0,1,4,0,0,8,11,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4356,Données client,OLD_7116545F,12,0,7,0,12,0,0,1,4,0,0,8,11,12
4357,Données client,OLD_7243183T,12,0,7,0,12,0,0,1,4,0,0,8,11,12
4358,Données client,OLD_7261535W,12,0,7,0,12,0,0,1,4,0,0,8,11,12
4359,Données client,OLD_7291299J,12,0,7,0,12,0,0,1,4,0,0,8,11,12


In [121]:
df[df.cluster_dist_20==6]

,provider_name,contractreference,FT1,FT2,FT3,FT4,FT5,FT6,FT7,FT8,FT9,FT10,FT11,cluster15,cluster20,cluster_dist_20
277,Veolia - 2,14156 003,9,0,8,0,9,0,0,1,0,0,0,4,6,6
298,Hidrogea,11141628,8,0,8,0,8,0,0,1,0,0,0,4,6,6
1936,CYO Eau de Cergy-Pontoise,8168114H,8,0,7,0,8,0,0,1,1,0,0,4,6,6
1937,Eau du Grand Lyon,1154716C,8,0,7,0,8,0,0,1,1,0,0,4,6,6
1938,ILEO,2192117H,8,0,7,0,8,0,0,1,1,0,0,4,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10808,Eaux du Vivier,00215013,8,0,8,0,8,0,0,1,0,0,0,4,6,6
10830,Eau de Marseille,7430477 X,9,0,7,0,9,0,0,1,1,0,0,4,6,6
10831,Cyo Direct,9341441 P,9,0,7,0,9,0,0,1,1,0,0,4,6,6
10837,Ileo,4072882 X,9,0,7,0,9,0,0,1,1,0,0,4,6,6


In [ ]:
# Des cluster parfaitement distincts
df["cluster_id"] = df.groupby(['FT1', 'FT2', 'FT3', 'FT4', 'FT5', 'FT6', 'FT7', 'FT8', 'FT9', 'FT10','FT11']).ngroup()
df.groupby(['provider_name', 'cluster_id'], as_index=False).contractreference.agg(['count', 'max']).sort_values('count')

In [148]:
grp_strict.columns = ['pattern_id', 'count_contractRefence', 'ex_contract_reference']

In [150]:
grp_strict.to_csv('./data/contract_reference_pattern.csv', index=False, sep=',')

In [154]:
df["contractreference"]= df["contractreference"].apply(lambda x: f'"{x}"')

In [163]:
df.groupby(['provider_name', 'cluster_id'], as_index=False).contractreference.agg(['count', 'max']).groupby('provider_name').count().sort_values('max').tail(10)

,cluster_id,count,max
provider_name,,,
Clermont Metropole 2,6,6,6
EXCEL HISTO CAGNE SUR MER,6,6,6
EXCEL - CELC,7,7,7
Veolia IDF,8,8,8
Colmarienne des eaux,8,8,8
Suez Eau France,11,11,11
Données Client,15,15,15
Veolia Eau,17,17,17
Veolia,23,23,23
